# Figure notebook — all datasets

Generates all paper figures for one or more questionnaire datasets.
Figures are saved to each dataset's own subfolder.

Data is loaded from `learning_cd_3rdattempt_miguelpc/{ds}-cd/` if available,
otherwise from `learning_cd_3rdattempt/{ds}-cd/` (automatic fallback).

In [12]:
import os, sys, pickle
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', message=".*timestamp seems very low.*")

# ── project paths ──────────────────────────────────────────────────────
REPO_ROOT = '/Users/ariannaarmanetti/Desktop/CODES/inverse-spin/'

import qspin
from qspin import (
    condition, model_label, bins,
    plot_losses,
    plot_moment_matching_learning,
    plot_moment_matching_sampling,
    plot_item_histogram,
    plot_E2d_histogram_maxent,
    plot_E2d_histogram_simple,
    plot_E2d_histogram_begvscopula,
    plot_mahalanobis_commonC_maxent,
    plot_mahalanobis_modelcov,
    plot_mahalanobis_commonC_simple,
    plot_mahalanobis_commonC_begvscopula,
    plot_pc_histogram_maxent,
    plot_pc_histogram_simple,
    plot_factor_histogram_maxent,
    plot_correlation_time_analysis
)
from qspin.nullmodels import (energy,
    catind_model, null_gaussian_copula, model_gaussdisc)

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Global style configuration

In [2]:
# colour palette
COLORS = dict(
    emp       = 'dimgrey',
    ising     = 'peru',
    bc        = 'lightcoral',
    beg       = 'brown',
    gauss     = 'cornflowerblue',
    gaussdisc = 'blueviolet',
    nullcat   = 'orchid',
    copula    = 'darkseagreen',
)

# plot configuration
CFG = dict(
    nbins_E2d      = 80,    # bins for Euclidean-distance histogram (model curves)
    nbins_energy   = 80,    # bins for Mahalanobis histogram (model curves)
    nbins_pc       = 60,    # bins for PC / factor histograms (model curves)
    nbins_emp      = 20,    # bins for the empirical bar
    mypvalue       = 0.05,  # Wilson CI confidence level
    alpha_emp      = 0.1,   # transparency of empirical histogram fill
    xlabel_fsize   = 15,
    suptitle_fsize = 16,
    nb_items       = 12,    # items shown in item histogram
    item_ncols     = 4,
)

matplotlib.rcParams['pdf.fonttype'] = 42   # editable text in PDFs
matplotlib.rcParams['ps.fonttype']  = 42


## Dataset selection and data paths

In [3]:
# Set DATASETS_TO_PROCESS to 'all' or a list, e.g. ['big5', 'gcbs']
#DATASETS_TO_PROCESS = 'all'
DATASETS_TO_PROCESS = 'acme'

ALL_DATASETS = [
    'big5', 'cfcs', 'dass', 'ei', 'gcbs',
    'hsns', 'iri', 'mach', 'pwe', 'rwas', 'sd3',
]

if DATASETS_TO_PROCESS == 'all':
    datasets = ALL_DATASETS
elif isinstance(DATASETS_TO_PROCESS, str):
    datasets = [DATASETS_TO_PROCESS]
else:
    datasets = list(DATASETS_TO_PROCESS)

# ── project paths ──────────────────────────────────────────────────────
REPO_ROOT = '/Users/ariannaarmanetti/Desktop/CODES/inverse-spin/'
# Fallback order: miguelpc first, then plain 3rdattempt
BASE_MIGUELPC  = REPO_ROOT + 'learning/'
#BASE_MIGUELPC   = REPO_ROOT + 'inverse_spinmodel/learning_cd_3rdattempt_miguelpc/'
BASE_3RDATTEMPT = REPO_ROOT + 'inverse_spinmodel/learning_cd_3rdattempt/'

def find_datapath(ds):
    """Return the subfolder that contains data.npz for this dataset."""
    for base in [BASE_MIGUELPC, BASE_3RDATTEMPT]:
        p = os.path.join(base, ds + '-cd')
        if os.path.isfile(os.path.join(p, 'data.npz')):
            return p
    raise FileNotFoundError('No data.npz found for dataset: ' + ds)

print('Datasets to process:', datasets)
for ds in datasets:
    try:
        print('  ' + ds.ljust(8) + ' -> ' + find_datapath(ds))
    except FileNotFoundError as err:
        print('  ' + ds.ljust(8) + ' -> NOT FOUND')


Datasets to process: ['acme']
  acme     -> /Users/ariannaarmanetti/Desktop/CODES/inverse-spin/learning/acme-cd


## Per-dataset settings

- `N_PCS`: number of principal components to plot (all datasets)
- `DO_FA` / `N_FA_COMPONENTS`: Factor Analysis (gcbs only)
- `fa_zoom_factor` / `pc_zoom`: indices (0-based) for zoom figures

In [5]:
# samples for simple-model generation
N_SIMPLEMODELS = 100_000

# PCs plotted for every dataset
N_PCS = 3

# PCs used for correlation-time analysis (needs 7 to match paper Fig 6)
N_PCS_ACF = 7

# Dataset-specific overrides
# Keys: do_fa, n_fa, fa_zoom_factor, pc_zoom
DATASET_CFG = {
    'gcbs': dict(do_fa=True, n_fa=6, fa_zoom_factor=1, pc_zoom=3),
}

def ds_cfg(ds, key, default=None):
    return DATASET_CFG.get(ds, {}).get(key, default)


In [9]:
COLORS = dict(
    emp       = 'dimgrey',    # dark grey 
    ising     = '#E69F00',  # arancio (alta luminanza)
    bc        = '#56B4E9',  # blu cielo (luminanza media)
    beg       = '#009E73',  # verde bluastro (scuro)
    gauss     = '#CC79A7',  # viola rossastro (medio)
    gaussdisc = '#D55E00',  # vermiglio (arancio scuro)
    nullcat   = '#0072B2',  # blu scuro 
    copula    = '#F0E442',  # giallo (OK con histtype='step' su sfondo bianco)
)

## Main loop

In [11]:
SEP = '=' * 60

for DATASET in datasets:
    print('')
    print(SEP)
    print('Processing: ' + DATASET)
    print(SEP)

    # 1. locate data
    datapath = find_datapath(DATASET)
    savepath = datapath   # figures saved alongside the data
    print('  datapath: ' + datapath)

    # 2. load data.npz
    data_dict = np.load(os.path.join(datapath, 'data.npz'))
    X   = data_dict['Xtrain']
    Xte = data_dict['Xtest']
    N, M = X.shape
    R = len(set(X.flatten().tolist()))
    item_bins = bins(R)
    print('  N=' + str(N) + '  M=' + str(M) + '  R=' + str(R))

    # 3. helper to load pickles from datapath
    def _load(fname):
        with open(os.path.join(datapath, fname), 'rb') as fh:
            return pickle.load(fh)

    # 4. training losses
    losses_ising = _load('losses_isingcd.pickle')
    losses_bc    = _load('losses_bccd.pickle')
    losses_beg   = _load('losses_begcd.pickle')

    # 5. learned models (PCD)
    inverseisingPCD = _load('inverseising_cd.pickle')
    inversebcPCD    = _load('inversebc_cd.pickle')
    inversebegPCD   = _load('inversebeg_cd.pickle')

    # 6. simulation dictionaries
    sim_dict_list_ising = _load('sim_dict_list_ising_cd.pickle')
    sim_dict_list_bc    = _load('sim_dict_list_bc_cd.pickle')
    sim_dict_list_beg   = _load('sim_dict_list_beg_cd.pickle')

    # 7. PCA on empirical data
    n_pcs     = N_PCS
    n_pcs_acf = N_PCS_ACF
    C = np.cov(X.T)
    eigvals, Udag = np.linalg.eig(C)
    idx   = np.argsort(eigvals)[::-1]
    # U_full: enough rows for both PC histograms and correlation-time analysis
    U_full = Udag.T[idx[:max(n_pcs, n_pcs_acf)]]
    U    = U_full[:n_pcs]    # used for PC histograms
    mu_X = np.mean(X, axis=0)

    for sdl in [sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg]:
        for el in sdl:
            if condition(el):
                el['PCs'] = (el['configurations'] - mu_X) @ U.T

    # 8. simple models
    mean_shift  = np.mean(range(1, R + 1))
    X_gauss     = np.random.multivariate_normal(mu_X, C, size=N_SIMPLEMODELS)
    X_catind    = catind_model(np.copy(X), R, N=N_SIMPLEMODELS)
    X_copula    = (null_gaussian_copula(
                       X_data=np.array(np.copy(X) + mean_shift, dtype=int),
                       N=N_SIMPLEMODELS, M=M, R=R, nfa=5)
                   - mean_shift)
    X_gaussdisc = model_gaussdisc(np.copy(X), R, N_SIMPLEMODELS) + 1 - mean_shift

    # Gaussian PCs (for PC histogram)
    Xprime_gauss = (X_gauss - np.mean(X_gauss, axis=0)) @ U.T

    # 9. Mahalanobis distances (common empirical C)
    CovX  = np.cov(X.T)
    meanX = np.mean(X, axis=0)
    _energy_fn = lambda x: energy(x, CovX, meanX)

    energies_X          = np.array([_energy_fn(x) for x in X])
    energies_Xgauss     = np.array([_energy_fn(x) for x in X_gauss])
    energies_Xcatind    = np.array([_energy_fn(x) for x in X_catind])
    energies_Xcopula    = np.array([_energy_fn(x) for x in X_copula])
    energies_Xgaussdisc = np.array([_energy_fn(x) for x in X_gaussdisc])

    energymax   = np.max(energies_X) * 1.1
    bins_energy = np.arange(1e-2, energymax, energymax / CFG['nbins_energy'])

    # 10. Factor Analysis (gcbs only, or any dataset with do_fa=True)
    do_fa       = ds_cfg(DATASET, 'do_fa', True)
    n_fa        = ds_cfg(DATASET, 'n_fa', 3)
    fa_zoom_idx = ds_cfg(DATASET, 'fa_zoom_factor', None)
    pc_zoom_idx = ds_cfg(DATASET, 'pc_zoom', None)

    if do_fa:
        from sklearn.decomposition import FactorAnalysis
        N_max_FA = 50_000
        fa = FactorAnalysis(n_components=n_fa)
        Xf = fa.fit_transform(X)
        for sdl in [sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg]:
            for el in sdl:
                if condition(el):
                    el['Yf'] = fa.transform(el['configurations'][:N_max_FA])
        print('  Factor Analysis: ' + str(n_fa) + ' components')

    # ================================================================
    # FIGURES
    # ================================================================
    print('  Generating figures...')

    # Fig 1 — training losses
    plot_losses(
        losses_ising, losses_bc, losses_beg,
        DATASET, savepath, COLORS)

    # Fig 2 — moment matching: learning phase
    plot_moment_matching_learning(
        inverseisingPCD, inversebcPCD, inversebegPCD,
        DATASET, savepath, COLORS, CFG)

    # Fig 3 — moment matching: sampling phase
    plot_moment_matching_sampling(
        X, sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        DATASET, savepath, COLORS, CFG)

    # Fig 4 — item histograms: maxent models
    plot_item_histogram(
        X, sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        item_bins, DATASET, savepath, COLORS, CFG)

    # Fig 5 — Euclidean distance: maxent + Gaussian
    # set X_gauss = None to skip Gaussian curve in this figure
    plot_E2d_histogram_maxent(
        X, sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        DATASET, savepath, COLORS, CFG,
        X_gauss=X_gauss)

    # Fig 6 — Euclidean distance: simple models
    plot_E2d_histogram_simple(
        X, X_catind, X_gauss, X_gaussdisc, X_copula,
        DATASET, savepath, COLORS, CFG)

    # Fig 7 — Euclidean distance: BEG vs copula
    plot_E2d_histogram_begvscopula(
        X, sim_dict_list_beg, X_copula,
        DATASET, savepath, COLORS, CFG)

    # Fig 8 — Mahalanobis (common C): maxent + Gaussian
    # set energies_Xgauss = None to skip Gaussian curve in this figure
    plot_mahalanobis_commonC_maxent(
        energies_X,
        sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        N, DATASET, savepath, COLORS, CFG,
        energies_Xgauss=energies_Xgauss)

    # Fig 9 — Mahalanobis (model-dependent Sigma): maxent
    plot_mahalanobis_modelcov(
        energies_X,
        sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        N, M, bins_energy, DATASET, savepath, COLORS, CFG)

    # Fig 10 — Mahalanobis (common C): simple models
    plot_mahalanobis_commonC_simple(
        energies_X, energies_Xcatind, energies_Xgauss,
        energies_Xgaussdisc, energies_Xcopula,
        bins_energy, N, M, DATASET, savepath, COLORS, CFG)

    # Fig 11 — Mahalanobis (common C): BEG vs copula
    plot_mahalanobis_commonC_begvscopula(
        energies_X, sim_dict_list_beg, energies_Xcopula,
        bins_energy, N, DATASET, savepath, COLORS, CFG)

    # Fig 12 — PC histograms: maxent + Gaussian
    # set Xprime_gauss = None to skip Gaussian curve in this figure
    plot_pc_histogram_maxent(
        X, sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        U, mu_X, n_pcs, DATASET, savepath, COLORS, CFG, eigvals,
        Xprime_gauss=Xprime_gauss)

    if pc_zoom_idx is not None:
        plot_pc_histogram_maxent(
            X, sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
            U, mu_X, n_pcs, DATASET, savepath, COLORS, CFG, eigvals,
            Xprime_gauss=Xprime_gauss,
            zoom=True, pcs_toplot=[pc_zoom_idx])

    # Fig 13 — PC histograms: simple models
    plot_pc_histogram_simple(
        X, X_catind, X_gauss, X_gaussdisc, X_copula,
        U, mu_X, n_pcs, DATASET, savepath, COLORS, CFG)

    # Fig 14 — factor histograms (only if do_fa is True)
    if do_fa:
        plot_factor_histogram_maxent(
            Xf,
            sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
            n_fa, DATASET, savepath, COLORS, CFG)
        if fa_zoom_idx is not None:
            plot_factor_histogram_maxent(
                Xf,
                sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
                n_fa, DATASET, savepath, COLORS, CFG,
                zoom=True, factors_to_plot=[fa_zoom_idx])

    # Figs 15-17 — correlation-time analysis
    plot_correlation_time_analysis(
        sim_dict_list_ising, sim_dict_list_bc, sim_dict_list_beg,
        U_full, n_pcs_acf, DATASET, savepath, COLORS)

    print('  Done -> ' + savepath)

print('')
print('All datasets processed.')



Processing: acme
  datapath: /Users/ariannaarmanetti/Desktop/CODES/inverse-spin/learning/acme-cd
  N=1009  M=36  R=5
  Factor Analysis: 3 components
  Generating figures...


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


  Done -> /Users/ariannaarmanetti/Desktop/CODES/inverse-spin/learning/acme-cd

All datasets processed.
